# Aliases de serialização

Às vezes queremos aliases **diferentes** na desserialização e na serialização.

Por exemplo: escrevemos uma API com um endpoint que consulta outra API e devolve os dados para quem chamou o nosso endpoint.

A API externa devolve nomes de campo que nem sempre seguem os padrões da nossa API. Então precisamos aceitar um nome ao receber os dados
da API externa, mas serializar com outro.

Por exemplo, a API externa pode devolver um JSON assim:

In [1]:
response_json = """
{
    "ID": 100,
    "FirstName": "Isaac",
    "lastname": "Newton"
}
"""

Quem escreveu essa API não se importou muito com convenções de nome! (Acontece bem mais do que se imagina.)

Precisamos lidar com isso na desserialização, mas queremos arrumar as coisas na serialização, para dar uma experiência melhor aos nossos usuários.

Ou seja: um alias para desserializar e outro para serializar (e os nomes dos campos continuam seguindo as convenções do Python).

Começamos com um modelo para a desserialização:

In [2]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

In [3]:
class Person(BaseModel):
    id_: int = Field(alias="ID")
    first_name: str = Field(alias="FirstName")
    last_name: str = Field(alias="lastname")

Com esse modelo, desserializamos o JSON:

In [4]:
p = Person.model_validate_json(response_json)
print(f"p: {p!r}")

p: Person(id_=100, first_name='Isaac', last_name='Newton')


Poderíamos permitir também o preenchimento pelo nome do campo, mas não é necessário aqui, então deixo de fora.

Se serializarmos uma instância de `Person`, usamos os nomes dos campos:

In [5]:
print(f"p.model_dump(): {p.model_dump()!r}")

p.model_dump(): {'id_': 100, 'first_name': 'Isaac', 'last_name': 'Newton'}


ou os aliases:

In [6]:
print(f"p.model_dump(by_alias=True): {p.model_dump(by_alias=True)!r}")

p.model_dump(by_alias=True): {'ID': 100, 'FirstName': 'Isaac', 'lastname': 'Newton'}


Mas esses aliases **não** são os que queremos expor.

Podemos sobrescrever o alias de serialização com o `Field`:

In [7]:
class Person(BaseModel):
    id_: int = Field(alias="ID", serialization_alias="id")
    first_name: str = Field(alias="FirstName", serialization_alias="firstName")
    last_name: str = Field(alias="lastname", serialization_alias="lastName")

In [8]:
p = Person.model_validate_json(response_json)

E agora, fazendo dump com aliases:

In [9]:
print(f"p.model_dump(by_alias=True): {p.model_dump(by_alias=True)!r}")

p.model_dump(by_alias=True): {'id': 100, 'firstName': 'Isaac', 'lastName': 'Newton'}


Os aliases de serialização foram usados.

O curso dizia que, na época da gravação, o Pydantic não permitia usar uma função para gerar aliases de serialização (só para aliases
comuns), obrigando a declarar o `serialization_alias` de cada campo.

> 📝 **Nota de revisão:** isso mudou. O `AliasGenerator` permite funções diferentes para validação e serialização. No exemplo, a entrada
> usa PascalCase e a saída camelCase. (Este exemplo específico ainda precisa de ajuste manual, porque a API externa é inconsistente — `ID`
> e `lastname` não seguem nenhum padrão; o gerador brilha quando os dois lados são **consistentes**.)

In [10]:
from pydantic import AliasGenerator
from pydantic.alias_generators import to_camel, to_pascal

class PersonGenerated(BaseModel):
    model_config = ConfigDict(
        alias_generator=AliasGenerator(validation_alias=to_pascal, serialization_alias=to_camel)
    )

    first_name: str
    last_name: str

In [11]:
pg = PersonGenerated.model_validate({"FirstName": "Isaac", "LastName": "Newton"})
print(f"pg.model_dump(by_alias=True): {pg.model_dump(by_alias=True)!r}")

pg.model_dump(by_alias=True): {'firstName': 'Isaac', 'lastName': 'Newton'}


Por fim, as informações dos campos de `Person`:

In [12]:
print(f"Person.model_fields: {Person.model_fields!r}")

Person.model_fields: {'id_': FieldInfo(annotation=int, required=True, alias='ID', alias_priority=2, serialization_alias='id'), 'first_name': FieldInfo(annotation=str, required=True, alias='FirstName', alias_priority=2, serialization_alias='firstName'), 'last_name': FieldInfo(annotation=str, required=True, alias='lastname', alias_priority=2, serialization_alias='lastName')}


Agora elas trazem tanto o alias quanto o alias de serialização.